# 08  -  End-to-End Pipeline

**Goal:** Run the complete experiment using `src/run_experiment.py`  -  from raw CSV to final test evaluation  -  inspect all generated artifacts, and understand how every preceding notebook fits into the orchestration.

**Module:** `src/run_experiment.py`

---

## 0  -  Imports

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Colab'da çalışıyorsa: Drive'daki toolkit.py'yi kullan
    from google.colab import drive
    drive.mount('/content/drive')

    TOOLKIT_PATH = '/content/drive/MyDrive/ANN-Project/toolkit.py'
    exec(open(TOOLKIT_PATH).read())
    setup()
    ROOT = Path('/content/repo')
else:
    # Lokal: ROOT proje köküdür (notebook'tan bir üst)
    ROOT = Path('..').resolve()
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.utils.config import load_config

CONFIG_PATH = ROOT / 'configs' / 'base.yaml'
config = load_config(str(CONFIG_PATH))
print('Config loaded successfully.')
print(f'Experiment name : {config["experiment"]["name"]}')
print(f'Enabled models  : {config["models"]["enabled"]}')

---
## 1 - Pipeline architecture overview

```text
run_experiment.py - main(config_path)
|
|- 1. load_config()                  <- configs/base.yaml
|- 2. set_global_seed(42)
|- 3. load_market_data()             <- data/raw/*.csv
|- 4. apply_missing_value_policy()   <- ffill_then_drop_head
|- 5. select_columns()               <- features + close + date
|- 6. split_dev_test()               <- 85% dev / 15% locked final test
|
|- Build config variants:
|   |- threshold_quantile_candidates <- q20/q30/q40/q50
|   |- lookback_candidates           <- 5/10/21/42
|
|- For each (variant, model, scaler) candidate:
|   |- make_expanding_folds()        <- chronological CV
|   |- compute_threshold()           <- fold train only
|   |- build_labeled_sequences_for_endpoints()
|   |   |- label endpoint stays inside split
|   |   |- feature window may use prior history
|   |- scale_sequence_data()         <- fit on train only
|   |- train model
|   |- choose_probability_threshold()<- internal validation MCC
|   |- evaluate validation fold
|   |- compute naive baselines
|
|- select_best_cv_result()           <- by CV MCC mean
|
|- run_final_train_and_test()
|   |- threshold and scaler fit from development only
|   |- selected probability threshold from development internal validation
|   |- final test evaluated once
|
|- Save artifacts:
    |- artifacts/metrics/{name}_report.json
    |- artifacts/predictions/{name}_test_predictions.csv
    |- experiments_summary.csv
```


---
## 2 - Enable the models and grids you want to run

Edit `configs/base.yaml` to control:

- `models.enabled`
- `preprocessing.scalers`
- `labeling.threshold_quantile_candidates`
- `sequence.lookback_candidates`
- `decision.threshold_search`

The default config now includes classical models, tree ensembles, GRU, and CNN1D so the comparison is broad enough for MCC/F1 diagnosis.


In [ ]:
print('Currently enabled models:')
for m in config['models']['enabled']:
    print(f'  - {m}')

print('\nCurrent label threshold quantile candidates:')
for q in config['labeling'].get('threshold_quantile_candidates', [config['labeling']['threshold_quantile']]):
    print(f'  - {q}')

print('\nCurrent lookback candidates:')
for lb in config['sequence'].get('lookback_candidates', [config['sequence']['lookback']]):
    print(f'  - {lb}')

print('\nThreshold search:')
print(config['decision'].get('threshold_search', {'enabled': False}))


---
## 3  -  Run the experiment

Aşağıdaki hücrede `RUN_MODELS` sözlüğünden istediğin modelleri `True` yap, gerisini `False` bırak.

**Colab'da:** toolkit otomatik clone + install + run yapar, sonuçları Drive'a kaydeder.
**Lokal:** doğrudan `run_experiment.main()` çağrılır.

In [ ]:
# ============================================================
#  MODEL SEÇİMİ — True/False ile aç/kapat
# ============================================================
RUN_MODELS = {
    'logreg'       : False,
    'random_forest': False,
    'xgboost'      : False,
    'lightgbm'     : False,
    'catboost'     : False,
    'gru'          : True,
    'lstm'         : True,
    'cnn1d'        : False,
    'tcn'          : False,
    'transformer'  : False,
}
# ============================================================

selected = [name for name, active in RUN_MODELS.items() if active]
if not selected:
    raise ValueError('En az bir model seçmelisin!')
print(f'Çalıştırılacak modeller: {selected}\n')

if IN_COLAB:
    # Toolkit zaten yüklendi (cell-2), run() çağır
    run(models=selected)
else:
    from src.run_experiment import main
    main(str(CONFIG_PATH), config_overrides={'models': {'enabled': selected}})

---
## 4  -  Inspect generated artifacts

After running the pipeline the following files are created:

In [ ]:
artifacts_root = ROOT / 'artifacts'

for subdir in ['metrics', 'predictions', 'models', 'plots']:
    folder = artifacts_root / subdir
    files  = list(folder.iterdir()) if folder.exists() else []
    print(f'artifacts/{subdir}/  ({len(files)} files)')
    for f in files:
        print(f'  {f.name}')

---
## 5  -  Load and display the experiment report

In [ ]:
artifacts_root = ROOT / 'artifacts'
metrics_dir = artifacts_root / 'metrics'
json_files = sorted(metrics_dir.glob('*_report.json'))

report = None
if json_files:
    report_path = json_files[-1]
    with open(report_path, encoding='utf-8') as f:
        report = json.load(f)

if IN_COLAB:
    show_results()
elif report is None:
    print('No reports found. Run the pipeline first (Section 3).')
else:
    print(f'Loading: {report_path.name}\n')

    best = report.get('best_cv_selection', {})
    final_test = report.get('final_test', {})
    metrics = final_test.get('metrics', {})

    print('=== Best CV Selection ===')
    print(f"  model   : {best.get('model_name')}")
    print(f"  scaler  : {best.get('scaler_name')}")
    print(f"  variant : {best.get('variant_tag', '-')}")
    print(f"  CV MCC  : {best.get('cv_summary', {}).get('mcc_mean')}")

    print('\n=== Final Test Results ===')
    for k in ['mcc', 'f1', 'balanced_accuracy', 'roc_auc', 'pr_auc']:
        print(f'  {k:<20}: {metrics.get(k)}')
    print(f"  selected_probability_threshold: {final_test.get('selected_probability_threshold')}")

    print('\n=== Baselines ===')
    for name, baseline_metrics in final_test.get('baselines', {}).items():
        print(f"  {name:<18}: MCC={baseline_metrics.get('mcc')} F1={baseline_metrics.get('f1')}")

---
## 5b - Tüm adaylar - CV MCC sıralaması

Pipeline `cv_candidates` listesi olarak her (model × scaler × quantile × lookback) kombinasyonunu kaydeder. Aşağıda hem genel top-20 hem de **her modelin kendi en iyi varyantı** görünür.

In [ ]:
if report is None:
    print('Report yok — önce pipeline çalıştır (Bölüm 3).')
else:
    rows = []
    for candidate in report.get('cv_candidates', []):
        cv = candidate.get('cv_summary', {})
        rows.append({
            'model'     : candidate.get('model_name'),
            'scaler'    : candidate.get('scaler_name'),
            'variant'   : candidate.get('variant_tag'),
            'cv_mcc'    : cv.get('mcc_mean'),
            'cv_mcc_std': cv.get('mcc_std'),
            'cv_f1'     : cv.get('f1_mean'),
            'cv_bal_acc': cv.get('balanced_accuracy_mean'),
            'cv_pr_auc' : cv.get('pr_auc_mean'),
        })

    if not rows:
        print('Raporda cv_candidates anahtarı yok.')
    else:
        cmp_df = (
            pd.DataFrame(rows)
              .sort_values('cv_mcc', ascending=False, na_position='last')
              .reset_index(drop=True)
        )

        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', 200)
        pd.set_option('display.float_format', lambda v: f'{v:.4f}' if pd.notna(v) else 'N/A')

        print(f'Toplam aday: {len(cmp_df)}\n')

        print('=== TOP 20 (genel) ===')
        print(cmp_df.head(20).to_string())

        print('\n=== EN İYİ VARYANT / MODEL ===')
        best_per_model = (
            cmp_df.dropna(subset=['cv_mcc'])
                  .drop_duplicates('model', keep='first')
                  .reset_index(drop=True)
        )
        print(best_per_model.to_string())

---
## 6  -  Load test predictions

In [ ]:
preds_dir  = artifacts_root / 'predictions'
pred_files = list(preds_dir.glob('*.csv'))

if not pred_files:
    print('No prediction files found. Run the pipeline first.')
else:
    preds_df = pd.read_csv(sorted(pred_files)[-1])
    print(f'Predictions shape: {preds_df.shape}')
    print(preds_df.head(10).to_string())

---
## 7  -  Experiments summary CSV

Every run appends a row to `experiments_summary.csv`. This provides a quick overview of all experiments performed.

In [ ]:
summary_csv = ROOT / 'experiments_summary.csv'

if summary_csv.exists():
    summary_df = pd.read_csv(summary_csv)
    print(f'Experiments logged: {len(summary_df)}')
    summary_df.sort_values('cv_mcc_mean', ascending=False).head(10)
else:
    print('experiments_summary.csv not found yet  -  it is created on the first run.')

---
## 8  -  Final test confusion matrix (when available)

In [ ]:
import seaborn as sns

if report is None:
    print('Run the pipeline to generate a real confusion matrix.')
else:
    cm = report.get('final_test', {}).get('metrics', {}).get('confusion_matrix')
    if cm:
        cm_arr = np.array(cm)
        labels = ['Bear (0)', 'Bull (1)']
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm_arr, annot=True, fmt='d', cmap='Blues',
                    xticklabels=labels, yticklabels=labels,
                    linewidths=0.5, ax=ax)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        ax.set_title('Final Test Set - Confusion Matrix', fontsize=12)
        plt.tight_layout()
        plt.show()
    else:
        print('Confusion matrix not found in report.')


---
## 9  -  Notebook series recap

| Notebook | Focus |
|---|---|
| 01 | Data loading, chronological order, feature visualisation |
| 02 | Preprocessing, forward returns, threshold, Bull/Bear labeling |
| 03 | Dev/test split, expanding CV folds, 3-D sequence tensors |
| 04 | Logistic Regression + 4 tree ensembles through CV |
| 05 | GRU / CNN1D / LSTM / TCN / Transformer architecture walkthrough |
| 06 | Deep learning training loop, LR search, early stopping |
| 07 | Metrics, confusion matrices, multi-model comparison |
| **08** | **End-to-end orchestration  -  `run_experiment.py`** |

---

**EHB 420E  -  Artificial Neural Networks, Istanbul Technical University, Spring 2026**